# [Analysis] MgF2 Substrate Metasurface - Spectral Analysis & Visualization

এই জুপিটার নোটবুকটি `dataset/spectral_data.csv` ডেটাসেটের অপটিক্যাল স্পেক্ট্রাম (Reflection Spectrum) **5.6 - 6.6 um** অপটিমাইজড রেঞ্জে বিশ্লেষণ ও ভিজ্যুয়ালাইজ করার জন্য প্রস্তুত করা হয়েছে।

---
### [Index] নোটবুকের সূচিপত্র:
* **[Section 0] Setup & Data Loading**: লাইব্রেরি ইমপোর্ট, ডেটাসেট লোড এবং ওয়েভলেংথ বিন্যাস।
* **[Section 1] Single Row Spectrum**: একটি নির্দিষ্ট রো/স্যাম্পলের জন্য ফুল স্পেক্ট্রাম এবং রেজোন্যান্স পিক ডিটেকশন।
* **[Section 2] Random 5 Spectra**: ডেটাসেট থেকে র্যান্ডম ৫টি রো নিয়ে তাদের ৫টি স্পেক্ট্রাম এক সাথে তুলনা।
* **[Section 3] 5 Parameters Sensitivity**: ৫টি প্যারামিটার (H, Px, Py, R1, R2) প্রতিটির পরিবর্তনের প্রভাব প্রদর্শনের জন্য ৫টি সাবপ্লট।
* **[Section 4] Standalone Parameter Explorer**: প্রতিটি প্যারামিটার আলাদা আলাদা ফুল-সাইজ প্লটে দেখার জন্য ফাংশন।


---
## [Section 0] Cell 0: লাইব্রেরি ইম্পোর্ট ও ডেটাসেট লোড
এখানে `dataset/spectral_data.csv` লোড করা হবে এবং অপটিমাইজড ওয়েভলেংথ রেঞ্জ (5.6 - 6.6 um) এসেন্ডিং অর্ডারে সাজানো হবে।

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

# Matplotlib সুন্দর ও প্রফেশনাল স্টাইলিং
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 14,
    'figure.dpi': 120,
    'lines.linewidth': 2.0,
    'grid.alpha': 0.4,
    'grid.linestyle': '--'
})

# ডেটাসেট পাথ
csv_path = os.path.join("dataset", "spectral_data.csv")
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"[ERROR] Dataset file not found: '{csv_path}'. Please run 'run_sweep.py' first.")

df = pd.read_csv(csv_path)
print("=" * 60)
print("[LOAD]  Dataset loaded successfully!")
print(f"[INFO]  Total samples (Rows): {len(df)} | Columns: {len(df.columns)}")

# ৫টি জিওমেট্রিক্যাল প্যারামিটার কলাম
geo_cols = ['H_um', 'Px_um', 'Py_um', 'R1_um', 'R2_um']
param_names = {
    'H_um': 'Pillar Height H',
    'Px_um': 'Periodicity Px',
    'Py_um': 'Periodicity Py',
    'R1_um': 'Ellipse Radius R1',
    'R2_um': 'Ellipse Radius R2'
}

# স্পেক্ট্রাল কলাম সমূহ (w_i এবং r_i)
w_cols = sorted([c for c in df.columns if c.startswith('w_')], key=lambda x: int(x.split('_')[1]))
r_cols = sorted([c for c in df.columns if c.startswith('r_')], key=lambda x: int(x.split('_')[1]))

# ওয়েভলেংথ এক্স-অক্ষ (Ascending অর্ডারে সাজানো: 5.6 um -> 6.6 um)
w_raw = df[w_cols].iloc[0].values.astype(float)
sort_idx = np.argsort(w_raw)
w_sorted = w_raw[sort_idx]

# অপটিমাইজড 5.6 থেকে 6.6 um রেঞ্জ ফিল্টার
valid_mask = (w_sorted >= 5.60) & (w_sorted <= 6.60)
wavelengths = w_sorted[valid_mask]

def get_spectrum(dataframe, row_idx):
    """নির্দিষ্ট রো এর জন্য সাজানো ওয়েভলেংথ এবং রিফ্লেকশন অ্যারে (5.6 - 6.6 um) প্রদান করে"""
    r_raw = dataframe[r_cols].iloc[row_idx].values.astype(float)
    r_sorted = r_raw[sort_idx]
    return wavelengths, r_sorted[valid_mask]

def get_param_str(dataframe, row_idx):
    """রো এর প্যারামিটারগুলোর একটি ফরম্যাটেড স্ট্রিং তৈরি করে"""
    row = dataframe.iloc[row_idx]
    return (f"H={row['H_um']:.3f} um, Px={row['Px_um']:.3f} um, "
            f"Py={row['Py_um']:.3f} um, R1={row['R1_um']:.3f} um, R2={row['R2_um']:.3f} um")

print(f"[RANGE] Wavelength band: {wavelengths.min():.2f} um to {wavelengths.max():.2f} um ({len(wavelengths)} points)")
print("=" * 60)
display(df[geo_cols].head())


---
## [Section 1] Cell 1: এক সেলে একটি রো-এর স্পেক্ট্রাম প্লট (Single Row Spectrum)
এই সেলে `row_idx` পরিবর্তন করে যেকোনো নির্দিষ্ট স্যাম্পলের রিফ্লেকশন স্পেক্ট্রাম (R vs lambda) দেখা যাবে।
- সর্বোচ্চ রেজোন্যান্স পিক চিহ্নিত করা হয়েছে।
- মেটাসারফেসের ৫টি প্যারামিটার একটি ইনফো বক্সে প্রদর্শিত হয়েছে।

In [ ]:
# যে রো দেখতে চান তার ইনডেক্স নির্ধারণ করুন (০ থেকে len(df)-1)
row_idx = 0

# স্পেক্ট্রাম ডেটা সংগ্রহ
w, r = get_spectrum(df, row_idx)
row_params = df.iloc[row_idx]

# রেজোন্যান্স পিক অনুসন্ধান
peaks, _ = find_peaks(r, height=0.2, distance=30)
if len(peaks) == 0:
  peak_idx = np.argmax(r)
  peaks = [peak_idx]
main_peak = peaks[np.argmax(r[peaks])]

# প্লট তৈরি
fig, ax = plt.subplots(figsize=(10, 5.5))

# মূল স্পেক্ট্রাম লাইন ও শেডিং
ax.plot(w, r, color='#1f77b4', linewidth=2.5, label='Reflection R(\\lambda)')
ax.fill_between(w, r, alpha=0.15, color='#1f77b4')

# পিক পয়েন্ট মার্কিং
peak_w = w[main_peak]
peak_r = r[main_peak]
ax.plot(peak_w, peak_r, marker='o', markersize=8, color='#d62728', label=f'Peak: \\lambda={peak_w:.3f} µm, R={peak_r:.3f}')
ax.axvline(peak_w, color='#d62728', linestyle=':', alpha=0.7)

# ইনফো বক্স (প্যারামিটারসমূহ)
info_text = (
  f"Structural Parameters:\n"
  f"• Height (H): {row_params['H_um']:.4f} µm\n"
  f"• Periodicity X (Px): {row_params['Px_um']:.4f} µm\n"
  f"• Periodicity Y (Py): {row_params['Py_um']:.4f} µm\n"
  f"• Radius 1 (R1): {row_params['R1_um']:.4f} µm\n"
  f"• Radius 2 (R2): {row_params['R2_um']:.4f} µm"
)
ax.text(0.97, 0.95, info_text, transform=ax.transAxes,
    fontsize=9.5, verticalalignment='top', horizontalalignment='right',
    bbox=dict(boxstyle='round,pad=0.6', facecolor='#f8f9fa', edgecolor='#cccccc', alpha=0.9))

# অক্ষ ও শিরোনাম
ax.set_title(f'Single Spectrum Analysis (5.6 - 6.6 µm) - Sample #{row_idx}', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Wavelength \\lambda (µm)', fontweight='bold')
ax.set_ylabel('Reflection R(\\lambda)', fontweight='bold')
ax.set_xlim(5.6, 6.6) # অপটিমাইজড ৫.৬ থেকে ৬.৬ µm
ax.set_ylim(-0.02, 1.05)
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(loc='upper left', frameon=True, framealpha=0.9)

plt.tight_layout()
plt.show()


---
## [Section 2] Cell 2: র‍্যান্ডম ৫টি রো নিয়ে স্পেক্ট্রাম তুলনা (Random 5 Spectra)
এই সেলে ডেটাসেট থেকে র্যান্ডম ৫টি রো বাছাই করে তাদের স্পেক্ট্রাম একই প্লটে বিভিন্ন রঙ ও লাইন স্টাইলে তুলনা করা হয়েছে।

In [ ]:
# ৫টি র্যান্ডম স্যাম্পল নির্বাচন
num_random = min(5, len(df))
np.random.seed(42)  # পুনরুৎপাদনের সুবিধার্থে সিড ব্যবহার
random_indices = np.random.choice(len(df), size=num_random, replace=False)

# কালার প্যালেট
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
line_styles = ['-', '--', '-.', ':', '-']

fig, ax = plt.subplots(figsize=(11, 6))

print("-" * 60)
print(f"[SELECT] Selected sample indices: {list(random_indices)}")
print("-" * 60)

for i, idx in enumerate(random_indices):
    w, r = get_spectrum(df, idx)
    row_p = df.iloc[idx]
    
    # পিক ফাইন্ডিং
    peak_idx = np.argmax(r)
    peak_w = w[peak_idx]
    peak_r = r[peak_idx]
    
    label = (f"Row {idx}: H={row_p['H_um']:.3f}, Px={row_p['Px_um']:.2f}, "
             f"Py={row_p['Py_um']:.2f}, R1={row_p['R1_um']:.2f}, R2={row_p['R2_um']:.2f} (Peak: {peak_w:.2f}um)")
    
    ax.plot(w, r, color=colors[i % len(colors)], linestyle=line_styles[i % len(line_styles)],
            linewidth=2.2, label=label)
    
    # পিকে মার্কার
    ax.scatter([peak_w], [peak_r], color=colors[i % len(colors)], s=50, zorder=5)

ax.set_title(f'Comparison of {num_random} Random Metasurface Spectra (5.6 - 6.6 um)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Wavelength \\lambda (um)', fontweight='bold')
ax.set_ylabel('Reflection R(\\lambda)', fontweight='bold')
ax.set_xlim(5.6, 6.6)
ax.set_ylim(-0.02, 1.05)
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(loc='upper right', frameon=True, framealpha=0.92, fontsize=8.5)

plt.tight_layout()
plt.show()

# নির্বাচিত ৫টি স্যাম্পলের প্যারামিটার টেবিল
print("[TABLE] Selected samples parameter details:")
display(df.loc[random_indices, geo_cols])


---
## [Section 3] Cell 3: ৫টি প্যারামিটারের প্রতিটির পরিবর্তনের জন্য ৫টি করে স্পেক্ট্রাম প্লট
মেটাসারফেস স্ট্রাকচারে ৫টি মুখ্য জিওমেট্রিক্যাল প্যারামিটার রয়েছে:
1. H (H_um): পিলার উচ্চতা (Pillar Height)
2. Px (Px_um): X-দিক বরাবর পিরিওডিসিটি (Periodicity X)
3. Py (Py_um): Y-দিক বরাবর পিরিওডিসিটি (Periodicity Y)
4. R1 (R1_um): ইলিপ্সের মাইনর রেডিয়াস (Radius 1)
5. R2 (R2_um): ইলিপ্সের মেজর রেডিয়াস (Radius 2)

প্রতিটি প্যারামিটারের জন্য আলাদা আলাদা একটি করে সাবপ্লট তৈরি করা হয়েছে।

In [ ]:
# ৫টি সাবপ্লটের একটি কমপ্রিহেন্সিভ ফিগার তৈরি
fig, axes = plt.subplots(nrows=5, ncols=1, figsize=(12, 18), sharex=True)

k_curves = min(5, len(df))
cmap = plt.cm.viridis

for ax_idx, param in enumerate(geo_cols):
  ax = axes[ax_idx]
  
  # নির্দিষ্ট প্যারামিটারের মান অনুসারে ডেটাফ্রেম সাজানো
  sorted_df = df.sort_values(by=param).reset_index(drop=True)
  
  # ৫টি সমদূরবর্তী স্যাম্পল নির্বাচন (Min to Max)
  selected_indices = np.linspace(0, len(sorted_df) - 1, k_curves, dtype=int)
  subset = sorted_df.iloc[selected_indices]
  
  for curve_i, (_, row) in enumerate(subset.iterrows()):
    w, r = get_spectrum(df, row.name if row.name in df.index else 0)
    val = row[param]
    
    # গ্রেডিয়েন্ট কালার
    color = cmap(curve_i / max(1, k_curves - 1))
    ax.plot(w, r, color=color, linewidth=2.0,
        label=f"{param} = {val:.4f} µm")
    
    # পিক পয়েন্ট মার্কিং
    p_idx = np.argmax(r)
    ax.scatter(w[p_idx], r[p_idx], color=color, s=35, zorder=4)
  
  ax.set_title(f'Effect of Parameter: {param_names[param]} ({param})', fontsize=12, fontweight='bold', loc='left')
  ax.set_ylabel('Reflection R(\\lambda)', fontweight='bold')
  ax.set_xlim(5.6, 6.6) # অপটিমাইজড ৫.৬ থেকে ৬.৬ µm
  ax.set_ylim(-0.02, 1.05)
  ax.grid(True, linestyle='--', alpha=0.5)
  ax.legend(loc='upper right', frameon=True, framealpha=0.9, fontsize=9, ncol=2)

# সর্বশেষ সাবপ্লটের নিচে এক্স-অক্ষ লেবেল
axes[-1].set_xlabel('Wavelength \\lambda (µm)', fontweight='bold', fontsize=12)
axes[-1].set_xlim(5.6, 6.6) # অপটিমাইজড ৫.৬ থেকে ৬.৬ µm

plt.suptitle('Parameter Sensitivity: Spectral Response (5.6 - 6.6 µm)', 
       fontsize=15, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()


---
## [Section 4] Cell 4: প্রতিটি প্যারামিটারের আলাদা আলাদা ইন্ডিভিজুয়াল প্লট (Standalone Plots)
যদি আপনি ৫টি সাবপ্লটের পরিবর্তে যেকোনো একটি প্যারামিটারের জন্য বড় সাইজের ডেডিকেটেড প্লট দেখতে চান, তবে এই সেলটি ব্যবহার করতে পারেন।

In [ ]:
def plot_parameter_effect(dataframe, target_param, num_curves=5, colormap='plasma'):
  """
  একটি নির্দিষ্ট প্যারামিটারের পরিবর্তনের প্রভাব বড় প্লটে প্রদর্শন করে (5.6 - 6.6 µm)।
  """
  k = min(num_curves, len(dataframe))
  sorted_df = dataframe.sort_values(by=target_param).reset_index(drop=True)
  selected_idx = np.linspace(0, len(sorted_df) - 1, k, dtype=int)
  subset = sorted_df.iloc[selected_idx]
  
  fig, ax = plt.subplots(figsize=(10, 5))
  cm = plt.get_cmap(colormap)
  
  for i, (_, row) in enumerate(subset.iterrows()):
    w, r = get_spectrum(df, row.name if row.name in df.index else 0)
    val = row[target_param]
    color = cm(i / max(1, k - 1))
    
    ax.plot(w, r, color=color, linewidth=2.2,
        label=f"{target_param} = {val:.4f} µm")
    
    # পিক মার্কিং
    peak_i = np.argmax(r)
    ax.scatter([w[peak_i]], [r[peak_i]], color=color, s=40, zorder=5)
  
  param_title = param_names.get(target_param, target_param)
  ax.set_title(f'Spectral Variation with Respect to {param_title} (5.6 - 6.6 µm)', fontsize=13, fontweight='bold', pad=10)
  ax.set_xlabel('Wavelength \\lambda (µm)', fontweight='bold')
  ax.set_ylabel('Reflection R(\\lambda)', fontweight='bold')
  ax.set_xlim(5.6, 6.6) # অপটিমাইজড ৫.৬ থেকে ৬.৬ µm
  ax.set_ylim(-0.02, 1.05)
  ax.grid(True, linestyle='--', alpha=0.5)
  ax.legend(loc='upper right', frameon=True, framealpha=0.9)
  plt.tight_layout()
  plt.show()

# ৫টি প্যারামিটারের প্রতিটির জন্য ৫টি আলাদা প্লট প্রদর্শন
for p in geo_cols:
  plot_parameter_effect(df, target_param=p, num_curves=5, colormap='plasma')
